# Single-exponential Hawkes: filtered CT-DDPG and exact HJB oracle

This notebook implements the classical additive Hawkes specification. The positive-part link is available only for a signed finite approximation; it is not applied to the true single-exponential intensity.

The static comparator is selected independently on validation paths and then frozen before the common test-path evaluation.

## Observed Filter State

The true environment is a one-dimensional single-exponential controlled Hawkes jump-diffusion. CT-DDPG uses the online state

`S_t = (t, X_t, Z^1_t, ..., Z^K_t)`, where `dZ^k_t = -rho_k Z^k_t dt + dN_t`.

The exact Hawkes decay is deliberately included as one member of the arithmetic filter bank, while the other filters are generic observed memory readouts. CT-DDPG does not use the kernel coefficient or intensity formula during learning. Its action is chosen from this filter state and drives the one observed closed-loop system; there is no counterfactual RL rollout controlled by hidden true memory.

## Model Description

The controlled state is

$$
dX_t=(b_0-\kappa X_t-b_a a_t)dt
+[\sigma_0+\sigma_xX_t+\sigma_a a_t]_+dW_t
+\gamma(t,X_{t-},a_t)dN_t.
$$

The true intensity is classical additive, with no global link and no total-intensity cap:

$$
\lambda_t^a=\mu(t,X_{t-},a_t)+\alpha q(a_t)Z_t^{\mathrm{exact}},
\qquad
q(a)=1-c_{\mathrm{eff}}\frac{a}{a_{1/2}+a}.
$$

Only a signed finite approximation would use

$$
\lambda_t^{K,a}=\mu(t,X_{t-},a_t)
+\left[\alpha q(a_t)\sum_{k=1}^K w_kZ_t^k\right]_+.
$$

The reward has no event-count term:

$$
r_t=-(c_xX_t^2+c_a a_t^2),
\qquad G(X_T)=-c_TX_T^2.
$$

The known-model DGM/HJB oracle uses the exact Markov state `(t, X, Z_exact)`. Lower `X` is not automatically better: the state cost is quadratic around zero, and control also changes drift, volatility, jump amplitude, and intensity. The baseline coefficient and jump amplitude are bounded, while total intensity remains uncapped. Since `alpha/decay=1.25/1.30<1`, this calibration is subcritical.

## Tunable Parameters

Start here. These parameters control the Hawkes environment, the LQ objective, CT-DDPG training, the DGM/HJB oracle, and the number of Monte Carlo paths used in the plots.

In [ ]:
PARAMS = {
    # Reproducibility and training length
    "seed": 11,
    "ct_episodes": 220,
    "warmup_episodes": 25,
    "updates_per_episode": 10,
    "eval_every": 20,
    "eval_episodes": 160,
    "ct_validation_seed": 2718,
    "validation_episodes": 300,
    "path_episodes": 20,
    "kernel_diagnostic_episodes": 40,
    "stability_episodes": 40,
    "static_actions": (0.39,),
    "static_selection_actions": (0.36, 0.37, 0.38, 0.39, 0.40, 0.41, 0.42),
    "static_selection_seed": 1731,
    "static_selection_episodes": 1200,
    "static_projection_step": 0.01,
    "force_reselect_static": False,
    "force_retrain_ct": False,
    "force_retrain_dgm": False,

    # Environment and objective
    "horizon": 5.0,
    "dt": 0.02,
    "discount_beta": 0.02,
    "base_intensity": 2.05,
    "state_intensity_sensitivity": 0.10,
    "action_intensity_sensitivity": -0.08,
    "baseline_min": 1e-6,
    "baseline_max": 6.0,
    "excitation": 1.25,
    "control_effect_strength": 1.25,
    "action_half_saturation": 0.35,
    "decay": 1.30,
    "filter_decays": (0.325, 0.650, 0.975, 1.300, 1.625, 1.950, 2.275, 2.600),
    "true_filter_index": 3,
    "jump_size": 0.10,
    "jump_size_x_strength": 0.015,
    "jump_size_action_strength": -0.015,
    "jump_size_min": 1e-4,
    "jump_size_max": 0.25,
    "drift_reversion": 0.45,
    "drift_constant": 0.02,
    "diffusion": 0.05,
    "volatility_x_strength": 0.015,
    "control_drift_strength": 0.72,
    "control_volatility_strength": 0.08,
    "state_cost": 0.80,
    "control_cost": 0.18,
    "terminal_x_cost": 0.60,

    # CT-DDPG actor/critic
    "hidden": (48, 48),
    "segment_len": 50,
    "batch_size": 128,
    "actor_batch_size": 256,
    "lr_actor": 2e-5,
    "lr_critic": 8e-4,
    "initial_action": 0.30,
    "actor_x_skip": 6.0,
    "actor_z_skip": 0.5,

    # DGM/HJB oracle
    "dgm_steps": 450,
    "dgm_batch_size": 256,
    "dgm_hidden_dim": 64,
    "dgm_layers": 3,
    "dgm_lr": 2e-3,
    "dgm_action_grid_size": 41,

    # Output folders
    "fig_dir": "figures",
    "out_dir": "results",
}

In [ ]:
# Quick visible check that this notebook is using the richer model version.
print("rich current-readout model parameters loaded")
print("filter_decays =", PARAMS["filter_decays"])
print("mu_x, mu_a =", PARAMS["state_intensity_sensitivity"], PARAMS["action_intensity_sensitivity"])
print("drift_constant =", PARAMS["drift_constant"])
print("volatility_x_strength =", PARAMS["volatility_x_strength"])
print("jump_size_x_strength, jump_size_action_strength =", PARAMS["jump_size_x_strength"], PARAMS["jump_size_action_strength"])


## Regenerated Results

The copied notebook outputs have been cleared so the figures and tables you see after rerunning are generated under the revised classical-positive-link model. The benchmark cells below write fresh logs to `results/` and PNG figures to `figures/` inside this experiment folder.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

preview_path = SECTION_DIR / "figures" / "policy_path_ensemble_ci.png" if "SECTION_DIR" in globals() else Path.cwd() / "figures" / "policy_path_ensemble_ci.png"
if preview_path.exists():
    display(Image(filename=str(preview_path)))
else:
    print("Run the training and plotting cells below to generate policy_path_ensemble_ci.png.")


## Imports

In [ ]:
%matplotlib inline
import csv
import json
import pickle
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
SECTION_DIR = next(
    (
        p
        for p in candidates
        if (p / "current_readout_env.py").exists()
    ),
    next(
        (
            p / "classical model with positive link" / "single_exponential_check_experiment"
            for p in candidates
            if (p / "classical model with positive link" / "single_exponential_check_experiment" / "current_readout_env.py").exists()
        ),
        cwd,
    ),
)
ROOT = next(
    (p for p in (SECTION_DIR, *SECTION_DIR.parents) if (p / "neural_numpy.py").exists() and (p / "replay.py").exists()),
    SECTION_DIR.parent,
)
for path in (ROOT, SECTION_DIR, SECTION_DIR.parent):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from current_readout_env import CurrentReadoutEnvConfig, SingleExpCurrentReadoutEnv
from ct_ddpg_current_readout import (
    AgentConfig,
    collect_episode,
    estimate_cost_to_go,
    evaluate_agent,
    train_current_readout,
)
from dgm_hjb_current_readout import DGMConfig, DGMHJBCurrentReadout, DGMOraclePolicy, train_dgm_oracle
from plotting_current_readout import (
    plot_benchmark_returns,
    plot_ct_history,
    plot_dgm_history,
    plot_episode_ensemble_comparison,
    plot_value_surface,
    realized_remaining_cost,
)
from stability_diagnostics import (
    plot_stability_certificate,
    sufficient_condition_report,
    uncapped_path_diagnostics,
)
from static_control_selection import select_static_control, plot_static_control_selection
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch: {torch.__version__}; DGM device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    DEVICE = "cpu"
    print(f"PyTorch check failed, using CPU: {exc}")
print(f"Experiment directory: {SECTION_DIR}")

## Build Configs

In [ ]:
fig_dir = SECTION_DIR / PARAMS["fig_dir"]
out_dir = SECTION_DIR / PARAMS["out_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

env_cfg = CurrentReadoutEnvConfig(
    horizon=PARAMS["horizon"],
    dt=PARAMS["dt"],
    beta=PARAMS["discount_beta"],
    base_intensity=PARAMS["base_intensity"],
    state_intensity_sensitivity=PARAMS["state_intensity_sensitivity"],
    action_intensity_sensitivity=PARAMS["action_intensity_sensitivity"],
    baseline_min=PARAMS["baseline_min"],
    baseline_max=PARAMS["baseline_max"],
    excitation=PARAMS["excitation"],
    control_effect_strength=PARAMS["control_effect_strength"],
    action_half_saturation=PARAMS["action_half_saturation"],
    decay=PARAMS["decay"],
    filter_decays=PARAMS["filter_decays"],
    true_filter_index=PARAMS["true_filter_index"],
    jump_size=PARAMS["jump_size"],
    jump_size_x_strength=PARAMS["jump_size_x_strength"],
    jump_size_action_strength=PARAMS["jump_size_action_strength"],
    jump_size_min=PARAMS["jump_size_min"],
    jump_size_max=PARAMS["jump_size_max"],
    drift_reversion=PARAMS["drift_reversion"],
    drift_constant=PARAMS["drift_constant"],
    diffusion=PARAMS["diffusion"],
    volatility_x_strength=PARAMS["volatility_x_strength"],
    control_drift_strength=PARAMS["control_drift_strength"],
    control_volatility_strength=PARAMS["control_volatility_strength"],
    state_cost=PARAMS["state_cost"],
    control_cost=PARAMS["control_cost"],
    terminal_x_cost=PARAMS["terminal_x_cost"],
)

agent_cfg = AgentConfig(
    hidden=PARAMS["hidden"],
    segment_len=PARAMS["segment_len"],
    batch_size=PARAMS["batch_size"],
    actor_batch_size=PARAMS["actor_batch_size"],
    lr_actor=PARAMS["lr_actor"],
    lr_critic=PARAMS["lr_critic"],
    initial_action=PARAMS["initial_action"],
    actor_x_skip=PARAMS["actor_x_skip"],
    actor_z_skip=PARAMS["actor_z_skip"],
)

dgm_cfg = DGMConfig(
    steps=PARAMS["dgm_steps"],
    batch_size=PARAMS["dgm_batch_size"],
    hidden_dim=PARAMS["dgm_hidden_dim"],
    num_layers=PARAMS["dgm_layers"],
    lr=PARAMS["dgm_lr"],
    action_grid_size=PARAMS["dgm_action_grid_size"],
    device=DEVICE,
    print_every=max(1, PARAMS["dgm_steps"] // 9),
)

env_cfg, agent_cfg, dgm_cfg

## Validation Selection of the Static Benchmark

The constant action is selected without using the final test paths. We estimate its cost on the declared validation grid, fit a local quadratic to reduce pathwise Monte Carlo noise, project the fitted minimizer to the `0.01` action grid, and then freeze that action for every comparison below.

In [ ]:
static_selection = select_static_control(
    env_cfg,
    evaluate_agent,
    PARAMS["static_selection_actions"],
    seed=PARAMS["static_selection_seed"],
    episodes=PARAMS["static_selection_episodes"],
    cache_path=out_dir / "static_control_selection.json",
    environment_signature=asdict(env_cfg),
    projection_step=PARAMS["static_projection_step"],
    force=PARAMS["force_reselect_static"],
)
selected_static_action = float(static_selection["selected_action"])
PARAMS["static_actions"] = (selected_static_action,)
print(
    f"Validation-selected static action: a={selected_static_action:.2f} "
    f"(continuous quadratic minimizer {static_selection['continuous_minimizer']:.4f})"
)
fig = plot_static_control_selection(
    static_selection,
    fig_dir / "static_control_selection.png",
    title="Single-exponential static-control validation profile",
)
plt.show()

## Train CT-DDPG

In [ ]:
ct_cache_path = out_dir / "ct_agent.pkl"
ct_metadata_path = out_dir / "ct_agent.json"
ct_signature = {
    "environment": asdict(env_cfg),
    "agent": asdict(agent_cfg),
    "training": {
        key: PARAMS[key]
        for key in (
            "seed", "ct_episodes", "warmup_episodes", "updates_per_episode",
            "eval_every", "eval_episodes", "ct_validation_seed"
        )
    },
}
ct_cache_valid = False
if ct_cache_path.exists() and ct_metadata_path.exists():
    stored = json.loads(ct_metadata_path.read_text(encoding="utf-8"))
    ct_cache_valid = json.loads(json.dumps(ct_signature, sort_keys=True)) == stored.get("signature")

if PARAMS["force_retrain_ct"] or not ct_cache_valid:
    agent, replay_buffer, ct_history, train_baselines = train_current_readout(
        seed=PARAMS["seed"],
        episodes=PARAMS["ct_episodes"],
        warmup_episodes=PARAMS["warmup_episodes"],
        updates_per_episode=PARAMS["updates_per_episode"],
        eval_every=PARAMS["eval_every"],
        eval_episodes=PARAMS["eval_episodes"],
        validation_seed=PARAMS["ct_validation_seed"],
        env_cfg=env_cfg,
        agent_cfg=agent_cfg,
    )
    with ct_cache_path.open("wb") as f:
        pickle.dump({"agent": agent, "history": ct_history, "baselines": train_baselines}, f)
    ct_metadata_path.write_text(
        json.dumps({"signature": json.loads(json.dumps(ct_signature, sort_keys=True))}, indent=2),
        encoding="utf-8",
    )
else:
    with ct_cache_path.open("rb") as f:
        cached = pickle.load(f)
    agent = cached["agent"]
    ct_history = cached["history"]
    train_baselines = cached["baselines"]
    replay_buffer = None
    print(f"Loaded compatible CT-DDPG cache from {ct_cache_path}")

ct_agent = agent
fig = plot_ct_history(ct_history, fig_dir / "ct_convergence.png")
plt.show()

## Train DGM/HJB Oracle

In [ ]:
dgm_cache_path = out_dir / "dgm_oracle.pt"
dgm_signature = {
    "environment": asdict(env_cfg),
    "config": asdict(dgm_cfg),
}
dgm_cache_valid = False
if dgm_cache_path.exists():
    checkpoint = torch.load(dgm_cache_path, map_location=DEVICE, weights_only=False)
    dgm_cache_valid = checkpoint.get("signature") == json.loads(json.dumps(dgm_signature, sort_keys=True))

if PARAMS["force_retrain_dgm"] or not dgm_cache_valid:
    oracle, dgm_history = train_dgm_oracle(env_cfg, dgm_cfg)
    torch.save(
        {
            "signature": json.loads(json.dumps(dgm_signature, sort_keys=True)),
            "state_dict": oracle.state_dict(),
            "history": dgm_history,
        },
        dgm_cache_path,
    )
else:
    oracle = DGMHJBCurrentReadout(env_cfg, dgm_cfg).to(DEVICE)
    oracle.load_state_dict(checkpoint["state_dict"])
    oracle.eval()
    dgm_history = checkpoint["history"]
    print(f"Loaded compatible DGM/HJB cache from {dgm_cache_path}")

# CUDA accelerates DGM training, but one-state sequential policy queries are
# faster on CPU because they are too small to amortize GPU launch overhead.
oracle = oracle.to("cpu")
oracle.eval()
oracle_policy = DGMOraclePolicy(oracle)
fig = plot_dgm_history(dgm_history, fig_dir / "dgm_hjb_convergence.png")
plt.show()

## Policy Benchmark

In [ ]:
rows = {
    "ct_ddpg": evaluate_agent(env_cfg, agent, seed=999, episodes=PARAMS["validation_episodes"]),
    "dgm_hjb_oracle": evaluate_agent(env_cfg, oracle_policy, seed=999, episodes=PARAMS["validation_episodes"]),
}
for static_action in PARAMS["static_actions"]:
    rows[f"constant_{static_action:.2f}"] = evaluate_agent(
        env_cfg,
        None,
        seed=999,
        episodes=PARAMS["validation_episodes"],
        constant_action=static_action,
    )

with (out_dir / "comparison.json").open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2)
with (out_dir / "ct_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(ct_history[-1].keys()))
    writer.writeheader()
    writer.writerows(ct_history)
with (out_dir / "dgm_history.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(dgm_history[-1].keys()))
    writer.writeheader()
    writer.writerows(dgm_history)

print(json.dumps(rows, indent=2))
fig = plot_benchmark_returns(rows, fig_dir / "benchmark_returns.png")
plt.show()


## 3D Cost-To-Go Surfaces

The fair CT-DDPG vs DGM comparison here is based on Monte Carlo cost-to-go under each policy, using the same simulator and cost definition. The DGM/HJB analytic surface is also plotted as the oracle PDE value.

The raw CT critic is kept only as a diagnostic. It is a learned reward critic, not an oracle HJB solution; away from the on-policy state distribution, especially at artificial `Z=0` states, it can be poorly calibrated. Do not rank policies from the raw CT critic surface.


In [ ]:
# CT-DDPG is shown by a policy Monte Carlo surface. The known-model DGM curve
# is the HJB/PDE solution itself; a second Monte Carlo surface would require a
# dense control-grid solve at every simulated time step and adds no new value.
fig = plot_value_surface(
    lambda times, states: estimate_cost_to_go(
        env_cfg, agent, times, states, seed=3100, rollouts=3
    ),
    env_cfg,
    save_path=fig_dir / "ct_mc_cost_surface_z0.png",
    title="CT-DDPG policy Monte Carlo cost-to-go surface, Z=0",
    z_level=0.0,
    grid_size=19,
)
plt.show()

fig = plot_value_surface(
    oracle.value_np,
    env_cfg,
    save_path=fig_dir / "dgm_hjb_cost_surface_z0.png",
    title="DGM/HJB oracle PDE cost surface U(t, X, Z=0)",
    z_level=0.0,
    grid_size=45,
)
plt.show()

fig = plot_value_surface(
    lambda times, states: -agent.value_path(times, states),
    env_cfg,
    save_path=fig_dir / "ct_raw_critic_diagnostic_z0.png",
    title="CT-DDPG raw critic diagnostic -V(t, X, Z=0), not a benchmark",
    z_level=0.0,
    grid_size=45,
)
plt.show()

## 20-Path Confidence-Interval Paths

In [ ]:
best_static_key = max(
    (key for key in rows if key.startswith("constant_")),
    key=lambda key: rows[key]["return_mean"],
)
best_static_action = float(best_static_key.split("_", 1)[1])
best_static_label = f"best static a={best_static_action:.2f}"

episode_groups = {"CT-DDPG": [], "DGM/HJB oracle": [], best_static_label: []}
value_path_groups = {key: [] for key in episode_groups}

for i in range(PARAMS["path_episodes"]):
    ensemble_seed = 8000 + i
    ct_ep = collect_episode(env_cfg, agent, np.random.default_rng(ensemble_seed), exploration_std=0.0)
    dgm_ep = collect_episode(env_cfg, oracle_policy, np.random.default_rng(ensemble_seed), exploration_std=0.0)
    static_ep = collect_episode(
        env_cfg,
        None,
        np.random.default_rng(ensemble_seed),
        constant_action=best_static_action,
    )
    episode_groups["CT-DDPG"].append(ct_ep)
    episode_groups["DGM/HJB oracle"].append(dgm_ep)
    episode_groups[best_static_label].append(static_ep)
    value_path_groups["CT-DDPG"].append(realized_remaining_cost(ct_ep, env_cfg))
    value_path_groups["DGM/HJB oracle"].append(realized_remaining_cost(dgm_ep, env_cfg))
    value_path_groups[best_static_label].append(realized_remaining_cost(static_ep, env_cfg))

fig = plot_episode_ensemble_comparison(
    episode_groups,
    value_path_groups,
    fig_dir / "policy_path_ensemble_ci.png",
    ci=0.90,
)
plt.show()


## Latest Figure Preview

This cell reloads the saved PNG version of the 20-path confidence-interval figure.

In [ ]:
display(Image(filename=str(fig_dir / "policy_path_ensemble_ci.png")))


## Kernel Approximation Diagnostic

This final diagnostic separates two signed approximations. The global signed fit approximates the single-exponential kernel on a dense lag grid; the on-policy signed fit regresses the realized Hawkes memory on CT-DDPG filter states along simulated paths. The arithmetic filter grid includes the true decay exactly, so both curves should recover the true kernel almost perfectly.


In [ ]:
import importlib
import kernel_diagnostics_current_readout as kdc

# Reload so a notebook kernel does not reuse an older diagnostic module.
kdc = importlib.reload(kdc)

fig, kernel_summary = kdc.plot_kernel_approximation_diagnostic(
    env_cfg,
    agent,
    save_path=fig_dir / "ct_kernel_approximation.png",
    episodes=PARAMS["kernel_diagnostic_episodes"],
    seed=5300,
)
print("global signed kernel-fit weights:", np.round(kernel_summary["global_signed_weights"], 4))
print("on-policy signed memory-fit weights:", np.round(kernel_summary["onpolicy_weights"], 4))
print({k: v for k, v in kernel_summary.items() if "weights" not in k})
plt.show()

# Proposition 3.6 is used as a sufficient-condition certificate. The stress
# panel runs the true single-exponential simulator without any intensity cap.
kernel_masses = kdc.stability_envelope_masses(env_cfg)
assumption_report = sufficient_condition_report(
    **kernel_masses,
    baseline_bound=env_cfg.baseline_max,
    jump_amplitude_bound=env_cfg.jump_size_max,
    p_star=4.0,
)
uncapped_summary = uncapped_path_diagnostics(
    lambda: SingleExpCurrentReadoutEnv(env_cfg),
    actions=(0.0, 0.3, 1.0),
    episodes=PARAMS["stability_episodes"],
    seed=7800,
)
print("Assumption 3.2 sufficient-condition report:")
print(json.dumps(assumption_report, indent=2))
for action in ("0.00", "0.30", "1.00"):
    print(action, {k: round(v, 4) for k, v in uncapped_summary[action].items()})
fig = plot_stability_certificate(
    assumption_report,
    uncapped_summary,
    save_path=fig_dir / "assumption_3_2_uncapped_diagnostic.png",
    title="Single-exponential experiment: subcritical certificate and uncapped paths",
)
plt.show()